In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

def read_table(table_name):
    return client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`").to_dataframe()

customers = read_table("customers")
categories = read_table("categories")
products = read_table("products")
orders = read_table("orders")
order_items = read_table("order_items")
payments = read_table("payments")
reviews = read_table("reviews")

customers.head()

c:\Users\ramir\Documents\GitHub\marzo\TeamChallenges\Team-Challenge-2---TB\parte_02\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date
0,120,Gioacchino,Pellegrini,oreste21@example.net,+33 (0)2 38 63 15 99,Netherlands,Amsterdam,affiliate,2025-06-03
1,163,Samir,Soffici,losarodolfo@example.org,+49(0) 838065787,Netherlands,Amsterdam,affiliate,2024-04-25
2,374,Roman,Ferrand,lversace@example.org,+39 0941737489,Netherlands,Amsterdam,affiliate,2023-11-11
3,141,Adamo,Sedano,joubertanastasie@example.org,+49(0) 745458929,Spain,Barcelona,affiliate,2025-10-17
4,158,Nicolas,Franco,moorefrank@example.com,02 51 85 37 47,Spain,Barcelona,affiliate,2025-04-19


In [2]:
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["shipped_at"] = pd.to_datetime(orders["shipped_at"], errors="coerce")
orders["delivered_at"] = pd.to_datetime(orders["delivered_at"], errors="coerce")
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
reviews["review_date"] = pd.to_datetime(reviews["review_date"], errors="coerce")
customers["registration_date"] = pd.to_datetime(customers["registration_date"], errors="coerce")

In [3]:
qa_results = []

def add_result(check_name, passed, detail):
    qa_results.append({
        "check_name": check_name,
        "passed": passed,
        "detail": detail
    })

# PK únicas
add_result("customers PK única", customers["customer_id"].is_unique, "customer_id debe ser único")
add_result("categories PK única", categories["category_id"].is_unique, "category_id debe ser único")
add_result("products PK única", products["product_id"].is_unique, "product_id debe ser único")
add_result("orders PK única", orders["order_id"].is_unique, "order_id debe ser único")
add_result("order_items PK única", order_items["order_item_id"].is_unique, "order_item_id debe ser único")
add_result("payments PK única", payments["payment_id"].is_unique, "payment_id debe ser único")
add_result("reviews PK única", reviews["review_id"].is_unique, "review_id debe ser único")

# FK válidas
add_result(
    "orders.customer_id válido",
    orders["customer_id"].isin(customers["customer_id"]).all(),
    "Todos los pedidos deben apuntar a un cliente existente"
)

add_result(
    "products.category_id válido",
    products["category_id"].isin(categories["category_id"]).all(),
    "Todos los productos deben apuntar a una categoría existente"
)

add_result(
    "order_items.order_id válido",
    order_items["order_id"].isin(orders["order_id"]).all(),
    "Todas las líneas deben apuntar a un pedido existente"
)

add_result(
    "order_items.product_id válido",
    order_items["product_id"].isin(products["product_id"]).all(),
    "Todas las líneas deben apuntar a un producto existente"
)

add_result(
    "payments.order_id válido",
    payments["order_id"].isin(orders["order_id"]).all(),
    "Todos los pagos deben apuntar a un pedido existente"
)

add_result(
    "reviews.order_item_id válido",
    reviews["order_item_id"].isin(order_items["order_item_id"]).all(),
    "Todas las reviews deben apuntar a una línea de pedido existente"
)

qa_df = pd.DataFrame(qa_results)
qa_df

,check_name,passed,detail
0,customers PK única,True,customer_id debe ser único
1,categories PK única,True,category_id debe ser único
2,products PK única,True,product_id debe ser único
3,orders PK única,True,order_id debe ser único
4,order_items PK única,True,order_item_id debe ser único
5,payments PK única,True,payment_id debe ser único
6,reviews PK única,True,review_id debe ser único
7,orders.customer_id válido,True,Todos los pedidos deben apuntar a un cliente e...
8,products.category_id válido,True,Todos los productos deben apuntar a una catego...
9,order_items.order_id válido,True,Todas las líneas deben apuntar a un pedido exi...


In [4]:
business_checks = []

def add_business_check(name, passed, detail):
    business_checks.append({
        "check_name": name,
        "passed": passed,
        "detail": detail
    })

# Cantidades positivas
add_business_check(
    "Cantidad positiva en order_items",
    (order_items["quantity"] > 0).all(),
    "Todas las cantidades deben ser mayores que 0"
)

# Precios positivos
add_business_check(
    "Unit price positivo",
    (order_items["unit_price"] > 0).all(),
    "Todos los precios unitarios deben ser mayores que 0"
)

# Descuentos razonables
add_business_check(
    "Discount entre 0 y 0.20",
    order_items["discount_pct"].between(0, 0.20).all(),
    "Los descuentos deben estar entre 0% y 20%"
)

# Totales de línea coherentes
calc_line_total = (order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount_pct"])).round(2)
add_business_check(
    "line_total correcto",
    (calc_line_total == order_items["line_total"].round(2)).all(),
    "El total de línea debe coincidir con quantity * unit_price * (1-discount)"
)

# Un pago por pedido
payments_per_order = payments.groupby("order_id").size()
add_business_check(
    "Un pago por pedido",
    (payments_per_order == 1).all() and len(payments_per_order) == len(orders),
    "Cada pedido debe tener exactamente un pago"
)

# Reviews solo sobre items comprados
reviewed_items_exist = reviews["order_item_id"].isin(order_items["order_item_id"]).all()
add_business_check(
    "Reviews sobre líneas existentes",
    reviewed_items_exist,
    "No debe haber reviews sobre líneas inexistentes"
)

business_df = pd.DataFrame(business_checks)
business_df

,check_name,passed,detail
0,Cantidad positiva en order_items,True,Todas las cantidades deben ser mayores que 0
1,Unit price positivo,True,Todos los precios unitarios deben ser mayores ...
2,Discount entre 0 y 0.20,True,Los descuentos deben estar entre 0% y 20%
3,line_total correcto,False,El total de línea debe coincidir con quantity ...
4,Un pago por pedido,True,Cada pedido debe tener exactamente un pago
5,Reviews sobre líneas existentes,True,No debe haber reviews sobre líneas inexistentes


In [5]:
monthly_revenue = (
    orders.merge(payments, on="order_id", how="inner")
    .query("payment_status == 'completed'")
    .assign(order_month=lambda df: df["order_date"].dt.to_period("M").astype(str))
    .groupby("order_month", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "monthly_revenue"})
    .sort_values("order_month")
)

monthly_revenue.head(12)

,order_month,monthly_revenue
0,2023-01,107828.88
1,2023-02,72740.34
2,2023-03,76726.09
3,2023-04,112919.68
4,2023-05,81292.17
5,2023-06,94699.47
6,2023-07,98791.76
7,2023-08,87696.66
8,2023-09,98938.69
9,2023-10,128150.24


In [6]:
top_products = (
    order_items.merge(products[["product_id", "product_name", "brand"]], on="product_id", how="left")
    .groupby(["product_id", "product_name", "brand"], as_index=False)
    .agg(
        total_units_sold=("quantity", "sum"),
        total_revenue=("line_total", "sum")
    )
    .sort_values(["total_units_sold", "total_revenue"], ascending=[False, False])
    .head(10)
)

top_products

,product_id,product_name,brand,total_units_sold,total_revenue
23,24,BassWave Mini 961,"Moore, Becker and Carlson",162,56382.24
35,36,Galaxy Nova 199,"Campos, Vaughn and Marquez",160,119683.43
57,58,Galaxy Tab Lite 308,"Cristoforetti, Bettin e Tomasetti SPA",156,66430.18
26,27,RGB Tower 345,Manufacturas Cantón y asociados S.Com.,156,38970.57
41,42,iTab Air 516,Ligorio s.r.l.,152,159590.42
15,16,Studio Max 246,Mocenigo SPA,149,15355.07
52,53,Galaxy Nova 175,Gehringer GmbH,147,107346.74
29,30,PulseBand X 652,Brunet Besson S.A.,147,73126.27
5,6,Pixel One 204,Morin S.A.R.L.,147,59367.36
21,22,Tab One 400,Orengo-Galilei e figli,146,59621.51


In [7]:
customers_by_country = (
    customers.groupby("country", as_index=False)
    .agg(total_customers=("customer_id", "count"))
    .sort_values("total_customers", ascending=False)
)

customers_by_country

,country,total_customers
2,Italy,116
3,Netherlands,112
1,Germany,96
0,France,92
4,Spain,84


In [8]:
delivery_time = orders[
    (orders["order_status"].isin(["delivered", "returned"])) &
    (orders["shipped_at"].notna()) &
    (orders["delivered_at"].notna())
].copy()

delivery_time["delivery_days"] = (delivery_time["delivered_at"] - delivery_time["shipped_at"]).dt.days

avg_delivery_by_country = (
    delivery_time.groupby("shipping_country", as_index=False)
    .agg(avg_delivery_days=("delivery_days", "mean"))
    .sort_values("avg_delivery_days")
)

avg_delivery_by_country

,shipping_country,avg_delivery_days
4,Spain,4.323171
0,France,4.406667
1,Germany,4.576923
2,Italy,4.577670
3,Netherlands,4.606965


In [9]:
rating_by_category = (
    reviews
    .merge(products[["product_id", "category_id"]], on="product_id", how="left")
    .merge(categories[["category_id", "category_name"]], on="category_id", how="left")
    .groupby("category_name", as_index=False)
    .agg(
        avg_rating=("rating", "mean"),
        total_reviews=("review_id", "count")
    )
    .sort_values(["avg_rating", "total_reviews"], ascending=[False, False])
)

rating_by_category

,category_name,avg_rating,total_reviews
5,Smartphones,3.870968,93
3,Peripherals,3.85,40
0,Audio,3.8,85
4,Smart Home,3.761905,21
6,Tablets,3.75,140
7,Wearables,3.726316,95
2,Laptops,3.628319,113
1,Gaming,3.611111,72


In [10]:
margin_by_category = (
    order_items.merge(products[["product_id", "category_id", "cost"]], on="product_id", how="left")
    .merge(categories[["category_id", "category_name"]], on="category_id", how="left")
    .assign(
        revenue=lambda df: df["line_total"],
        estimated_cost=lambda df: df["quantity"] * df["cost"],
        margin=lambda df: df["line_total"] - (df["quantity"] * df["cost"])
    )
    .groupby("category_name", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        total_cost=("estimated_cost", "sum"),
        total_margin=("margin", "sum")
    )
    .sort_values("total_margin", ascending=False)
)

margin_by_category

,category_name,total_revenue,total_cost,total_margin
2,Laptops,2190295.64,1538000.96,652294.68
6,Tablets,976554.55,727353.46,249201.09
5,Smartphones,799069.94,658507.11,140562.83
7,Wearables,439031.11,315979.85,123051.26
1,Gaming,329516.89,261528.54,67988.35
0,Audio,190824.65,147314.08,43510.57
3,Peripherals,80827.16,58665.15,22162.01
4,Smart Home,19840.45,14737.25,5103.2
